# PID stabilization of the inverted rotary-bar pendulum

Around upright, define `theta = 0` at the inverted equilibrium. The acceleration-input plant is

\[
\ddot\theta-a\theta=bu,\qquad u=\ddot\phi,
\]

\[
P(s)=\frac{\Theta(s)}{U(s)}=\frac{b}{s^2-a}.
\]

The controller uses

\[
e=0-\theta,
\qquad
u=K_pe+K_i\int e\,dt+K_d\dot e.
\]

The closed-loop characteristic polynomial becomes

\[
s^3+bK_ds^2+(bK_p-a)s+bK_i.
\]

We choose desired poles \(-4,-5,-6\) and match coefficients. This is a simple linear design intended to show expected behavior, not a final hardware tuning.

In [1]:
from pathlib import Path
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

g = 9.81
l = 0.235
r = 0.14
a = 3*g/(2*l)
b = 3*r/(2*l)

desired_poles = np.array([-4.0, -5.0, -6.0])
poly = np.poly(desired_poles)  # [1, c2, c1, c0]
Kd = poly[1] / b
Kp = (poly[2] + a) / b
Ki = poly[3] / b

print(f'Plant: Theta/U = {b:.6f} / (s^2 - {a:.6f})')
print(f'open-loop unstable pole = +{np.sqrt(a):.6f} 1/s')
print(f'Kp threshold a/b = g/r = {a/b:.6f}')
print(f'PID gains from poles {desired_poles.tolist()}:')
print(f'  Kp = {Kp:.6f}')
print(f'  Ki = {Ki:.6f}')
print(f'  Kd = {Kd:.6f}')

Plant: Theta/U = 0.893617 / (s^2 - 62.617021)
open-loop unstable pole = +7.913092 1/s
Kp threshold a/b = g/r = 70.071429
PID gains from poles [-4.0, -5.0, -6.0]:
  Kp = 152.880952
  Ki = 134.285714
  Kd = 16.785714


## Simulation assumptions

- initial upright error: \(\theta(0)=5^\circ\)
- initial angular velocity: zero
- reference: \(\theta_{ref}=0\)
- commanded arm acceleration is saturated at \(\pm30\,\mathrm{rad/s^2}\)
- the model remains linear and neglects motor lag, missed steps, friction, and the previously neglected velocity-squared coupling terms.

The simulation also integrates \(u=\ddot\phi\) so that the expected rotary-arm velocity and position are visible.

In [2]:
theta0 = np.deg2rad(5.0)
u_max = 30.0  # rad/s^2
t_end = 5.0

# State x = [theta, theta_dot, integral_error, phi, phi_dot]
def rhs(t, x):
    theta, theta_dot, integ_e, phi, phi_dot = x
    e = -theta
    e_dot = -theta_dot
    u_unsat = Kp*e + Ki*integ_e + Kd*e_dot
    u = np.clip(u_unsat, -u_max, u_max)
    theta_ddot = a*theta + b*u
    return [theta_dot, theta_ddot, e, phi_dot, u]

t_eval = np.linspace(0, t_end, 5001)
sol = solve_ivp(rhs, (0, t_end), [theta0,0,0,0,0], t_eval=t_eval,
                max_step=0.001, rtol=1e-9, atol=1e-11)

t = sol.t
theta, theta_dot, integ_e, phi, phi_dot = sol.y
e = -theta
e_dot = -theta_dot
u_unsat = Kp*e + Ki*integ_e + Kd*e_dot
u = np.clip(u_unsat, -u_max, u_max)

# simple 0.1 deg band settling-time estimate
band = np.deg2rad(0.1)
settling_time = np.nan
for i in range(len(t)):
    if np.all(np.abs(theta[i:]) <= band):
        settling_time = t[i]
        break

print(f'peak |u| = {np.max(np.abs(u)):.4f} rad/s^2')
print(f'min theta = {np.rad2deg(np.min(theta)):.4f} deg')
print(f'final theta = {np.rad2deg(theta[-1]):.6f} deg')
print(f'settling time into +/-0.1 deg band = {settling_time:.4f} s')
print(f'final phi_dot = {phi_dot[-1]:.6f} rad/s')
print(f'final phi = {np.rad2deg(phi[-1]):.3f} deg')

peak |u| = 13.3414 rad/s^2
min theta = -1.2219 deg
final theta = -0.000000 deg
settling time into +/-0.1 deg band = 1.6230 s
final phi_dot = -0.000000 rad/s
final phi = -49.390 deg


In [3]:
fig, axes = plt.subplots(4, 1, figsize=(9, 9), sharex=True)
axes[0].plot(t, np.rad2deg(theta))
axes[0].axhline(0, linewidth=0.8)
axes[0].set_ylabel('theta [deg]')
axes[0].grid(True)

axes[1].plot(t, u)
axes[1].axhline(u_max, linestyle='--', linewidth=0.8)
axes[1].axhline(-u_max, linestyle='--', linewidth=0.8)
axes[1].set_ylabel('u = phi_ddot\n[rad/s^2]')
axes[1].grid(True)

axes[2].plot(t, phi_dot)
axes[2].set_ylabel('phi_dot [rad/s]')
axes[2].grid(True)

axes[3].plot(t, np.rad2deg(phi))
axes[3].set_ylabel('phi [deg]')
axes[3].set_xlabel('time [s]')
axes[3].grid(True)

fig.suptitle('Inverted pendulum: PID stabilization from 5 deg initial tilt')
fig.tight_layout()
fig_path = Path('figures/inverted_pid_response.svg')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'saved: {fig_path}')

saved: figures/inverted_pid_response.svg


![Inverted PID response](figures/inverted_pid_response.svg)

## Expected behavior and hardware caution

The PID makes the acceleration command oppose the unstable gravitational divergence. For the default 5 degree initial error, the simulated pendulum returns to upright and the arm acceleration returns near zero.

However, this PID regulates only `theta`. It does **not** explicitly regulate the rotary-arm angle `phi`, so the arm can finish at a nonzero angular position after the balancing maneuver. A practical rotary inverted pendulum often uses full-state feedback (for example LQR using \(\theta,\dot\theta,\phi,\dot\phi\)) or an additional arm-position objective.

For a stepper motor, `u` must also be limited by the available torque-speed envelope. Integrate `u` to obtain the commanded arm velocity and convert that velocity to pulse frequency. If the demanded acceleration is too high, missed steps invalidate the assumed relation between command and actual \(\phi\).